# Langbridge SDK Federated Query Notebook

This walkthrough shows the local Langbridge runtime federating three live sources at query time:

- a Postgres sales database
- a Postgres CRM database
- a local CSV with campaign attribution tags

The shared key is `contact_external_id`, so the runtime can combine revenue, CRM context, and marketing campaign data without copying them into one warehouse first.


In [15]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display

EXAMPLE_DIR = Path.cwd().resolve()
if not (EXAMPLE_DIR / "langbridge_config.yml").exists():
    candidate = EXAMPLE_DIR / "examples" / "sdk" / "federated_query"
    if candidate.exists():
        EXAMPLE_DIR = candidate.resolve()

REPO_ROOT = EXAMPLE_DIR.parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from langbridge import LangbridgeClient


## Start the Postgres sources

This example uses Docker to start the sales and CRM databases from the seed scripts in this folder.


In [13]:
subprocess.run(["docker", "compose", "up", "-d", "--wait"], cwd=EXAMPLE_DIR, check=True)
subprocess.run(["docker", "compose", "ps"], cwd=EXAMPLE_DIR, check=True)


 Container federated-crm-db  Running
 Container federated-sales-db  Running
 Container federated-crm-db  Waiting
 Container federated-sales-db  Waiting
 Container federated-crm-db  Healthy
 Container federated-sales-db  Healthy


NAME                 IMAGE                COMMAND                  SERVICE    CREATED          STATUS                    PORTS
federated-crm-db     postgres:16-alpine   "docker-entrypoint.s…"   crm-db     11 minutes ago   Up 10 minutes (healthy)   0.0.0.0:5433->5432/tcp, [::]:5433->5432/tcp
federated-sales-db   postgres:16-alpine   "docker-entrypoint.s…"   sales-db   11 minutes ago   Up 10 minutes (healthy)   0.0.0.0:5432->5432/tcp, [::]:5432->5432/tcp


CompletedProcess(args=['docker', 'compose', 'ps'], returncode=0)

In [18]:
client = LangbridgeClient.local(config_path=str(EXAMPLE_DIR / "langbridge_config.yml"))
datasets = client.datasets.list()
dataset_ids = {item.name: item.id for item in datasets.items}

datasets_df = pd.DataFrame([item.model_dump(mode="json") for item in datasets.items])
datasets_df = datasets_df.sort_values("name").reset_index(drop=True)
datasets_df


,id,name,label,description,connector,semantic_model,materialization_mode,managed,semantic_models,materialization,source,schema_hint,sync,status,sync_status,last_sync_at,management_mode
0,12d9bcee-b0bc-5689-afc1-cf150fab64b3,crm_contacts,CRM Contacts,CRM contacts enriched with account segment and...,crm_reporting,customer_revenue_federation,live,True,[customer_revenue_federation],{'mode': 'live'},"{'kind': 'sql', 'sql': 'SELECT 'CRM-' || LPA...",None,None,published,None,None,config_managed
1,e367228d-1980-5ff1-8623-d63f9938d24a,marketing_campaigns,Marketing Campaigns,Campaign metadata loaded from a local CSV and ...,campaign_file,customer_revenue_federation,live,True,[customer_revenue_federation],{'mode': 'live'},"{'kind': 'file', 'storage_uri': 'file:///home/...",None,None,published,None,None,config_managed
2,60ff3235-1b12-501d-a9bc-43ccc84129a0,sales_orders,Sales Orders,Order-level sales facts enriched with the CRM ...,sales_reporting,customer_revenue_federation,live,True,[customer_revenue_federation],{'mode': 'live'},"{'kind': 'sql', 'sql': 'SELECT o.id AS order...",None,None,published,None,None,config_managed


In [19]:
def preview_dataset(name: str, limit: int = 5) -> pd.DataFrame:
    result = client.datasets.query(dataset_id=dataset_ids[name], limit=limit)
    return pd.DataFrame(result.rows)

for dataset_name in ["sales_orders", "crm_contacts", "marketing_campaigns"]:
    print(dataset_name)
    display(preview_dataset(dataset_name))


sales_orders


,order_id,crm_contact_external_id,country,loyalty_tier,channel,status,order_date,order_total
0,1,CRM-00000001,US,bronze,store,returned,2026-03-06,794.83
1,2,CRM-00000001,US,bronze,store,returned,2026-03-06,3005.47
2,3,CRM-00000002,US,bronze,store,completed,2025-09-03,1860.36
3,4,CRM-00000002,US,bronze,store,completed,2026-03-12,472.38
4,5,CRM-00000003,US,bronze,online,processing,2025-11-11,1094.84


crm_contacts


,contact_external_id,account_segment,owner_region,lifecycle_stage,lead_score,preferred_channel,marketing_opt_in,last_touch_date
0,CRM-00000001,startup,West,prospect,95,email,True,2025-12-14
1,CRM-00000002,smb,North,prospect,51,email,True,2025-12-14
2,CRM-00000003,enterprise,Metro,customer,53,phone,True,2026-03-02
3,CRM-00000004,mid_market,Coastal,customer,19,chat,True,2026-03-13
4,CRM-00000005,smb,Enterprise,customer,26,chat,True,2026-03-01


marketing_campaigns


,contact_external_id,campaign_name,campaign_channel,offer_type,target_region,touch_date,engagement_score
0,CRM-00000001,Spring Refresh,email,discount,East,2026-01-05,91
1,CRM-00000002,Spring Refresh,email,discount,East,2026-01-06,87
2,CRM-00000003,Spring Refresh,email,discount,East,2026-01-06,84
3,CRM-00000004,Spring Refresh,email,discount,East,2026-01-07,79
4,CRM-00000005,Spring Refresh,email,discount,East,2026-01-08,88


## Federated semantic query

This semantic query asks for revenue from the sales database, grouped by CRM account segment and CSV campaign name.


In [20]:
semantic_result = client.semantic.query(
    "customer_revenue_federation",
    measures=["sales_orders.net_revenue", "sales_orders.order_count"],
    dimensions=["marketing_campaigns.campaign_name", "crm_contacts.account_segment"],
    filters=[{"member": "marketing_campaigns.campaign_name", "operator": "set"}],
    order={"sales_orders.net_revenue": "desc"},
    limit=12,
)

pd.DataFrame(semantic_result.rows)


,marketing_campaigns.campaign_name,crm_contacts.account_segment,sales_orders.net_revenue,sales_orders.order_count
0,Premium Upgrade,smb,25205.05,18
1,Summer Bundle,mid_market,17804.21,14
2,Reactivation Push,mid_market,17288.39,14
3,Reactivation Push,startup,17067.88,12
4,Spring Refresh,smb,16504.22,12
5,Store Launch,smb,16457.33,12
6,VIP Retention,enterprise,14640.71,11
7,VIP Retention,smb,14604.57,12
8,Store Launch,startup,14091.88,10
9,Summer Bundle,startup,13756.06,9


## Dataset SQL across runtime datasets

Here the runtime uses dataset SQL scope, federates across the eligible workspace datasets, and derives the logical SQL aliases from dataset metadata.


In [ ]:
sql_result = client.sql.query(
    query_scope="dataset",
    query="""
    SELECT
      m.campaign_name,
      c.account_segment,
      COUNT(DISTINCT s.order_id) AS order_count,
      ROUND(SUM(s.order_total), 2) AS net_revenue
    FROM sales_orders AS s
    JOIN crm_contacts AS c
      ON s.crm_contact_external_id = c.contact_external_id
    JOIN marketing_campaigns AS m
      ON c.contact_external_id = m.contact_external_id
    GROUP BY m.campaign_name, c.account_segment
    ORDER BY net_revenue DESC, order_count DESC
    LIMIT 5
    """,
)

pd.DataFrame(sql_result.rows)

,campaign_name
0,Store Launch
1,Summer Bundle
2,Premium Upgrade
3,Reactivation Push
4,Spring Refresh


When you are done, run `docker compose down -v` from this folder to remove the demo databases.


In [11]:
result = client.agents.ask("Which marketing campaign generated the most revenue from enterprise customers? and which month was it?")
print(result)

thread_id=UUID('53d343db-d1e8-4b50-81e6-cf436d420614') status='succeeded' job_id=UUID('47b678fa-c6ff-4aec-8e6e-22848543ea6d') summary='The top enterprise revenue result was $1,439,943.70 in February 2026, but the campaign name is null, so no specific marketing campaign can be identified from the verified result.' result={'columns': ['campaign_name', 'order_date', 'net_revenue'], 'rows': [[None, '2026-02-01', '1439943.70']], 'rowcount': 1, 'elapsed_ms': None, 'source_sql': 'SELECT t2."campaign_name" AS "marketing_campaigns__campaign_name", DATE_TRUNC(\'MONTH\', CAST(t0."order_date" AS TIMESTAMP)) AS "sales_orders__order_date_month", SUM(t0."order_total") AS "sales_orders__net_revenue" FROM sales_orders AS t0 LEFT JOIN crm_contacts AS t1 ON t0.crm_contact_external_id = t1.contact_external_id LEFT JOIN marketing_campaigns AS t2 ON t2.contact_external_id = t1.contact_external_id WHERE t1."account_segment" = \'enterprise\' GROUP BY t2."campaign_name", DATE_TRUNC(\'MONTH\', CAST(t0."order_da